## Stats SA Data Extraction Approach

Before extracting the data, the Stats SA SuperWEB platform will be inspected to understand how the available municipal and statistical data is accessed.

The extraction will focus on **KwaZulu-Natal** and will investigate the datasets available for municipalities, including Census 2022 Municipal Profiles, time-series municipal profiles, Community Survey data, health and vital statistics, social and labour statistics, and municipal boundary information.

Where possible, the extraction will obtain data that can be linked to municipalities and wards and that can support the analysis of voter participation and the characteristics of areas with different levels of participation.

The extraction process will first establish access to the Stats SA platform, identify the available datasets and variables, and then retrieve the relevant data. Raw extracted data will be preserved in `data/raw/statssa/`.

This notebook is limited to **data extraction and verification**. Cleaning, transformation, feature engineering, analysis, and modelling will be handled later in the project.

In [1]:
# We are extracting data from Statistics South Africa (Stats SA)
# for KwaZulu-Natal.
#
# The main source is the Stats SA SuperWEB platform:
# https://superweb.statssa.gov.za
#
# We will investigate available datasets including:
# - Census 2022 Municipal Profiles
# - Time Series Municipal Profiles
# - Community Survey
# - Health and Vital Statistics
# - Social Statistics
# - Labour Statistics
# - Municipal boundaries
#
# Raw extracted data will be saved to:
# data/raw/statssa/

from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "statssa"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Stats SA source
STATSSA_URL = "https://superweb.statssa.gov.za"

# Extraction scope
TARGET_PROVINCE = "KwaZulu-Natal"

# Historical period of interest
START_YEAR = 2011
CURRENT_YEAR = 2026

print("Stats SA extraction setup complete.")
print("Source:", STATSSA_URL)
print("Province:", TARGET_PROVINCE)
print("Period:", START_YEAR, "to", CURRENT_YEAR)
print("Raw directory:", RAW_DIR)

Stats SA extraction setup complete.
Source: https://superweb.statssa.gov.za
Province: KwaZulu-Natal
Period: 2011 to 2026
Raw directory: c:\Users\Admin\Desktop\Dirisa_1\SPU-TEAM-DIRISA\data\raw\statssa


In [3]:
# We first open the Stats SA SuperWEB page and keep a session active.
# The session is needed because Stats SA SuperWEB may use
# cookies and session state when accessing the available data.

statssa_session = requests.Session()

initial_response = statssa_session.get(
    STATSSA_URL,
    timeout=30
)

print("HTTP status:", initial_response.status_code)
print("Content type:", initial_response.headers.get("Content-Type"))
print("Response size:", len(initial_response.content), "bytes")

assert initial_response.status_code == 200, (
    "Stats SA SuperWEB page could not be loaded."
)

initial_soup = BeautifulSoup(
    initial_response.text,
    "html.parser"
)

print("Stats SA source loaded successfully.")
print("Session established successfully.")


HTTP status: 200
Content type: text/html
Response size: 212 bytes
Stats SA source loaded successfully.
Session established successfully.


In [4]:
# Check what the Stats SA server actually returned.
# This helps us confirm whether we reached the SuperWEB platform,
# a login page, or a redirect/information page before extraction.

print(initial_response.text)

<html>
<head>
<META NAME="robots" CONTENT="noindex,nofollow">
<script src="/_Incapsula_Resource?SWJIYLWA=5074a744e2e3d891814e9a2dace20bd4,719d34d31c8e3a6e6fffd425f7e032f3">
</script>
<body>
</body></html>



In [6]:
# ============================================
# CELL 3 — INSPECT STATS SA FORM FIELDS
# ============================================

# Stats SA SuperWEB uses session-based web requests.
# We need to identify the hidden fields and other
# session information generated by the page before
# attempting to reproduce any data requests.
#
# These values are inspected dynamically because
# they may change between sessions.

form_state = {}

for field in initial_soup.select(
    "input[type='hidden'][name]"
):
    name = field.get("name")
    value = field.get("value", "")

    form_state[name] = value

print("Hidden form fields found:", len(form_state))

print("\nHidden fields:")
for field in form_state:
    print("-", field)

Hidden form fields found: 0

Hidden fields:


In [7]:
# ============================================
# CELL 4 — IDENTIFY STATS SA DATA CONTROLS
# ============================================

# We now inspect the available form controls on the
# Stats SA SuperWEB page.
#
# The goal is to identify how the platform represents
# dataset, province, geography, or table selections
# before sending a request.
#
# We do not invent request parameters. We first inspect
# the controls returned by Stats SA.

select_fields = initial_soup.select("select[name]")

print("Select fields found:", len(select_fields))

print("\nAvailable selection controls:")

for field in select_fields:
    print(
        "Name:", field.get("name"),
        "| ID:", field.get("id")
    )

    options = field.find_all("option")

    print("Options:", len(options))

    for option in options[:10]:
        print(
            "  ",
            option.get("value", ""),
            "→",
            option.get_text(strip=True)
        )

    print()

Select fields found: 0

Available selection controls:


In [8]:
# ============================================
# CELL 4 — ACCESS STATS SA DATA CATALOGUE
# ============================================

# The browser Network activity showed that Stats SA
# retrieves its available database tables through a
# REST GET request.
#
# We reproduce that request programmatically so that
# we can inspect the datasets available through SuperWEB.

CATALOGUE_URL = (
    "https://superweb.statssa.gov.za/"
    "webapi/rest/catalogue/databaseTables/tree"
)

catalogue_response = statssa_session.get(
    CATALOGUE_URL,
    params={"nocache": "1"},
    timeout=30
)

print("HTTP status:", catalogue_response.status_code)
print("Content type:", catalogue_response.headers.get("Content-Type"))
print("Response size:", len(catalogue_response.content), "bytes")

assert catalogue_response.status_code == 200, (
    "Stats SA catalogue could not be loaded."
)

print("\nResponse preview:")
print(catalogue_response.text[:1000])

HTTP status: 200
Content type: text/html
Response size: 212 bytes

Response preview:
<html>
<head>
<META NAME="robots" CONTENT="noindex,nofollow">
<script src="/_Incapsula_Resource?SWJIYLWA=5074a744e2e3d891814e9a2dace20bd4,719d34d31c8e3a6e6fffd425f7e032f3">
</script>
<body>
</body></html>



In [9]:
# ============================================
# CELL 5 — Reproduce the Stats SA JSF request
# ============================================

# Stats SA uses JSF AJAX requests for the Data Catalogue.
# We reproduce the POST request structure observed in Chrome.

STATSSA_JSF_URL = (
    "https://superweb.statssa.gov.za"
    "/webapi/jsf/dataCatalogueExplorer.xhtml"
)

jsf_payload = {
    "j_id_44_SUBMIT": "1",
    "jakarta.faces.behavior.event": "action",
    "jakarta.faces.source": "j_id_4a",
    "jakarta.faces.partial.ajax": "true",
    "j_id_44": "j_id_44",
    "jakarta.faces.partial.execute": "j_id_4a",
    "jakarta.faces.partial.render": (
        "databasesPanelButtons "
        "tablesPanelButtons "
        "updateTablesTree "
        "infoPanel "
        "messagePanel"
    )
}

jsf_headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Faces-Request": "partial/ajax",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": STATSSA_JSF_URL,
}

jsf_response = statssa_session.post(
    STATSSA_JSF_URL,
    data=jsf_payload,
    headers=jsf_headers,
    timeout=30
)

print("HTTP status:", jsf_response.status_code)
print("Content type:", jsf_response.headers.get("Content-Type"))
print("Response size:", len(jsf_response.content), "bytes")

print("\nFirst 1000 characters of response:")
print(jsf_response.text[:1000])

HTTP status: 200
Content type: text/html
Response size: 212 bytes

First 1000 characters of response:
<html>
<head>
<META NAME="robots" CONTENT="noindex,nofollow">
<script src="/_Incapsula_Resource?SWJIYLWA=5074a744e2e3d891814e9a2dace20bd4,719d34d31c8e3a6e6fffd425f7e032f3">
</script>
<body>
</body></html>



In [10]:
# ============================================
# CELL 6 — Test the Stats SA REST API
# ============================================

# Chrome successfully accessed this endpoint.
# We test whether Python can access the same API.

STATSSA_TREE_URL = (
    STATSSA_URL
    + "/webapi/rest/catalogue/databaseTables/tree"
)

tree_response = statssa_session.get(
    STATSSA_TREE_URL,
    params={"nocache": "test"},
    headers={
        "Accept": "*/*",
        "Referer": STATSSA_JSF_URL,
    },
    timeout=30
)

print("HTTP status:", tree_response.status_code)
print("Content type:", tree_response.headers.get("Content-Type"))
print("Response size:", len(tree_response.content), "bytes")

print("\nResponse:")
print(tree_response.text[:1000])

HTTP status: 200
Content type: text/html
Response size: 212 bytes

Response:
<html>
<head>
<META NAME="robots" CONTENT="noindex,nofollow">
<script src="/_Incapsula_Resource?SWJIYLWA=5074a744e2e3d891814e9a2dace20bd4,719d34d31c8e3a6e6fffd425f7e032f3">
</script>
<body>
</body></html>



In [11]:
# ============================================
# CELL 5 — Stats SA API Authentication
# ============================================

# Enter your Stats SA Open Data API key here.
# Keep this key private and DO NOT commit it to GitHub.

STATSSA_API_KEY = "65794a30655841694f694a4b563151694c434a68624763694f694a49557a49314e694a392e65794a7063334d694f694a7a644849756333526c6247786863694973496e4e3159694936496d786c614778765a323975623278766257393061476c696155426e625746706243356a623230694c434a70595851694f6a45334f5441794f5445794e7a5573496d46315a434936496e4e30636935765a47456966512e4e3076582d73776d51506b444b6461613545356d733567496d4f4b324953326443517a3658694152466f49"

# Basic verification without displaying the key
assert STATSSA_API_KEY != "PASTE_YOUR_API_KEY_HERE", (
    "65794a30655841694f694a4b563151694c434a68624763694f694a49557a49314e694a392e65794a7063334d694f694a7a644849756333526c6247786863694973496e4e3159694936496d786c614778765a323975623278766257393061476c696155426e625746706243356a623230694c434a70595851694f6a45334f5441794f5445794e7a5573496d46315a434936496e4e30636935765a47456966512e4e3076582d73776d51506b444b6461613545356d733567496d4f4b324953326443517a3658694152466f49"
)

print("Stats SA API key loaded successfully.")

Stats SA API key loaded successfully.


In [12]:
# ============================================
# CELL 6 — Test Stats SA Open Data API
# ============================================

STATSSA_API_URL = (
    "https://superweb.statssa.gov.za/webapi/api/v1"
)

api_test_response = requests.get(
    STATSSA_API_URL,
    headers={
        "APIKey": STATSSA_API_KEY,
        "Accept": "application/json"
    },
    timeout=30
)

print("HTTP status:", api_test_response.status_code)
print("Content type:", api_test_response.headers.get("Content-Type"))
print("Response size:", len(api_test_response.content), "bytes")

print("\nResponse preview:")
print(api_test_response.text[:500])

HTTP status: 200
Content type: text/html
Response size: 212 bytes

Response preview:
<html>
<head>
<META NAME="robots" CONTENT="noindex,nofollow">
<script src="/_Incapsula_Resource?SWJIYLWA=5074a744e2e3d891814e9a2dace20bd4,719d34d31c8e3a6e6fffd425f7e032f3">
</script>
<body>
</body></html>



In [14]:
# ============================================
# FIND TABLE1 TO TABLE5
# ============================================

TABLES_DIR = Path.home() / "Desktop" / "Dirisa_1"

print("Tables folder:", TABLES_DIR)

table_files = sorted(TABLES_DIR.glob("TABLE*"))

print("\nTables found:")

for file in table_files:
    print("-", file.name)

Tables folder: C:\Users\Admin\Desktop\Dirisa_1

Tables found:
- TABLE1.xlsx
- TABLE2.xlsx
- TABLE3.xlsx
- TABLE4.xlsx
- TABLE5.xlsx


In [16]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------

In [17]:
# ============================================
# INSPECT TABLE1 TO TABLE5
# ============================================

for file in table_files:

    df = pd.read_excel(file)

    print(f"\n{'=' * 50}")
    print(file.name)
    print(f"{'=' * 50}")

    print("Rows:", len(df))
    print("Columns:")
    
    for column in df.columns:
        print(" -", column)

    print("\nFirst 3 rows:")
    display(df.head(3))


TABLE1.xlsx
Rows: 15
Columns:
 - Unnamed: 0
 - Unnamed: 1
 - Unnamed: 2
 - Unnamed: 3
 - Unnamed: 4
 - Unnamed: 5
 - Unnamed: 6
 - Unnamed: 7
 - Unnamed: 8
 - Unnamed: 9
 - Unnamed: 10
 - Unnamed: 11
 - Unnamed: 12
 - Unnamed: 13
 - Unnamed: 14
 - Unnamed: 15
 - Unnamed: 16
 - Unnamed: 17
 - Unnamed: 18
 - Unnamed: 19
 - Unnamed: 20
 - Unnamed: 21
 - Unnamed: 22
 - Unnamed: 23
 - Unnamed: 24
 - Unnamed: 25
 - Unnamed: 26
 - Unnamed: 27
 - Unnamed: 28
 - Unnamed: 29
 - Unnamed: 30
 - Unnamed: 31
 - Unnamed: 32
 - Unnamed: 33
 - Unnamed: 34
 - Unnamed: 35
 - Unnamed: 36
 - Unnamed: 37
 - Unnamed: 38
 - Unnamed: 39
 - Unnamed: 40
 - Unnamed: 41
 - Unnamed: 42
 - Unnamed: 43
 - Unnamed: 44
 - Unnamed: 45
 - Unnamed: 46
 - Unnamed: 47
 - Unnamed: 48
 - Unnamed: 49
 - Unnamed: 50
 - Unnamed: 51
 - Unnamed: 52
 - Unnamed: 53
 - Unnamed: 54
 - Unnamed: 55
 - Unnamed: 56
 - Unnamed: 57
 - Unnamed: 58
 - Unnamed: 59
 - Unnamed: 60
 - Unnamed: 61
 - Unnamed: 62
 - Unnamed: 63
 - Unnamed: 64
 - U

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 504,Unnamed: 505,Unnamed: 506,Unnamed: 507,Unnamed: 508,Unnamed: 509,Unnamed: 510,Unnamed: 511,Unnamed: 512,Unnamed: 513
0,Descriptive 2022 (MN Phase 2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Province, district and municipality and Questi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Counting: Population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



TABLE2.xlsx
Rows: 70
Columns:
 - Unnamed: 0
 - Unnamed: 1

First 3 rows:


,Unnamed: 0,Unnamed: 1
0,Descriptive 2022 (MN Phase 2),NaN
1,"Province, district and municipality",NaN
2,Counting: Population,NaN



TABLE3.xlsx
Rows: 173
Columns:
 - Unnamed: 0
 - Unnamed: 1

First 3 rows:


,Unnamed: 0,Unnamed: 1
0,Descriptive 2022 (MN Phase 2),NaN
1,"Province, district and municipality then Sex t...",NaN
2,Counting: Population,NaN



TABLE4.xlsx
Rows: 14
Columns:
 - Unnamed: 0
 - Unnamed: 1
 - Unnamed: 2
 - Unnamed: 3
 - Unnamed: 4
 - Unnamed: 5
 - Unnamed: 6
 - Unnamed: 7
 - Unnamed: 8
 - Unnamed: 9
 - Unnamed: 10
 - Unnamed: 11
 - Unnamed: 12
 - Unnamed: 13
 - Unnamed: 14
 - Unnamed: 15
 - Unnamed: 16
 - Unnamed: 17
 - Unnamed: 18
 - Unnamed: 19
 - Unnamed: 20
 - Unnamed: 21
 - Unnamed: 22
 - Unnamed: 23
 - Unnamed: 24
 - Unnamed: 25
 - Unnamed: 26
 - Unnamed: 27
 - Unnamed: 28
 - Unnamed: 29
 - Unnamed: 30
 - Unnamed: 31
 - Unnamed: 32
 - Unnamed: 33
 - Unnamed: 34
 - Unnamed: 35
 - Unnamed: 36
 - Unnamed: 37
 - Unnamed: 38
 - Unnamed: 39
 - Unnamed: 40
 - Unnamed: 41
 - Unnamed: 42
 - Unnamed: 43
 - Unnamed: 44
 - Unnamed: 45
 - Unnamed: 46
 - Unnamed: 47
 - Unnamed: 48
 - Unnamed: 49
 - Unnamed: 50
 - Unnamed: 51
 - Unnamed: 52
 - Unnamed: 53
 - Unnamed: 54
 - Unnamed: 55
 - Unnamed: 56
 - Unnamed: 57
 - Unnamed: 58
 - Unnamed: 59
 - Unnamed: 60
 - Unnamed: 61
 - Unnamed: 62
 - Unnamed: 63
 - Unnamed: 64
 - U

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 88,Unnamed: 89,Unnamed: 90,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97
0,Descriptive 2022 (MN Phase 2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Province, district and municipality then Atten...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Counting: Population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



TABLE5.xlsx
Rows: 14
Columns:
 - Unnamed: 0
 - Unnamed: 1
 - Unnamed: 2
 - Unnamed: 3
 - Unnamed: 4
 - Unnamed: 5
 - Unnamed: 6
 - Unnamed: 7
 - Unnamed: 8
 - Unnamed: 9
 - Unnamed: 10
 - Unnamed: 11
 - Unnamed: 12
 - Unnamed: 13
 - Unnamed: 14
 - Unnamed: 15
 - Unnamed: 16
 - Unnamed: 17
 - Unnamed: 18
 - Unnamed: 19
 - Unnamed: 20

First 3 rows:


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Descriptive 2022 (MN Phase 2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Province, district and municipality then Quest...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Counting: Population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# ============================================
# INSPECT TABLE5 RAW EXCEL LAYOUT
# ============================================

table5 = TABLES_DIR / "TABLE5.xlsx"

raw_table5 = pd.read_excel(
    table5,
    header=None
)

print("Rows:", len(raw_table5))
print("Columns:", len(raw_table5.columns))

display(raw_table5)

Rows: 15
Columns: 21


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Descriptive 2022 (MN Phase 2),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Province, district and municipality then Quest...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Counting: Population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Filters:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Default Summation,Population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"Province, district and municipality then Quest...",KwaZulu-Natal,Household,Homeless,Transient,Institution,Urban area,Tribal or Traditional area,Farm area,Formal residential,...,Traditional residential,Farms,Parks and recreation,Collective living quarters,Industrial,Small holdings,Vacant,Commercial,Male,Female
9,NaN,12423906.662782,61367658.619297,55719,2623,601502,39366168.367817,20254463.904037,2406870.347507,34773150.614651,...,19811611.317867,1643269.858092,24476.10274,288493.911735,81909.621183,266590.178248,66941.948922,218006.301069,30078756.687159,31948745.932222


In [19]:
# ============================================
# INSPECT RAW STATS SA TABLE STRUCTURE
# ============================================

for file in table_files:

    print(f"\n{'=' * 60}")
    print(file.name)
    print(f"{'=' * 60}")

    raw_df = pd.read_excel(
        file,
        header=None
    )

    print("Shape:", raw_df.shape)

    # Show all non-empty cells
    for row_index, row in raw_df.iterrows():

        values = [
            str(value)
            for value in row
            if pd.notna(value)
        ]

        if values:
            print(f"Row {row_index}:")
            print(values[:10], "..." if len(values) > 10 else "")


TABLE1.xlsx
Shape: (16, 514)
Row 1:
['Descriptive 2022 (MN Phase 2)'] 
Row 2:
['Province, district and municipality and Questionnaire type then Geography type then Sex'] 
Row 3:
['Counting: Population'] 
Row 5:
['Filters:'] 
Row 6:
['Default Summation', 'Population'] 
Row 8:
['Province, district and municipality', 'KwaZulu-Natal', 'Ugu', 'Umdoni Local Municipality', 'Umzumbe Local Municipality', 'UMuziwabantu Local Municipality', 'Ray Nkonyeni Local Municipality', 'Umgungundlovu', 'uMshwathi Local Municipality', 'uMngeni Local Municipality'] ...
Row 9:
['Questionnaire type then Geography type then Sex', 'Household', 'Homeless', 'Transient', 'Institution', 'Urban area', 'Tribal or Traditional area', 'Farm area', 'Male', 'Female'] ...
Row 10:
['12279374.662785625', '7768', '93', '136671', '6060765.093845516', '5819613.969936846', '543527.5990030895', '5919216.636794859', '6504690.025991391', '771411.6543514762'] ...
Row 15:
['© Statistics South Africa'] 

TABLE2.xlsx
Shape: (71, 2)
Row 

In [20]:
# ============================================
# SUMMARISE STATS SA TABLE STRUCTURE
# ============================================

for file in table_files:

    raw_df = pd.read_excel(
        file,
        header=None
    )

    print(f"\n{'=' * 60}")
    print(file.name)
    print(f"{'=' * 60}")

    print("Shape:", raw_df.shape)

    # Show the important metadata rows
    for row_number in [1, 2, 3, 6]:
        if row_number < len(raw_df):
            values = raw_df.iloc[row_number].dropna().tolist()

            print(f"\nRow {row_number}:")
            print(values[:10])

    # Show first 10 geography headings
    if len(raw_df) > 8:
        geography = raw_df.iloc[8].dropna().tolist()

        print("\nGeography/classification row:")
        print(geography[:15])

    # Show first 10 values from the data row
    if len(raw_df) > 10:
        data_values = raw_df.iloc[10].dropna().tolist()

        print("\nFirst data row:")
        print(data_values[:15])


TABLE1.xlsx
Shape: (16, 514)

Row 1:
['Descriptive 2022 (MN Phase 2)']

Row 2:
['Province, district and municipality and Questionnaire type then Geography type then Sex']

Row 3:
['Counting: Population']

Row 6:
['Default Summation', 'Population']

Geography/classification row:
['Province, district and municipality', 'KwaZulu-Natal', 'Ugu', 'Umdoni Local Municipality', 'Umzumbe Local Municipality', 'UMuziwabantu Local Municipality', 'Ray Nkonyeni Local Municipality', 'Umgungundlovu', 'uMshwathi Local Municipality', 'uMngeni Local Municipality', 'Mpofana Local Municipality', 'Impendle Local Municipality', 'The Msunduzi Local Municipality', 'Mkhambathini Local Municipality', 'Richmond Local Municipality']

First data row:
[12279374.662785625, 7768, 93, 136671, 6060765.093845516, 5819613.969936846, 543527.5990030895, 5919216.636794859, 6504690.025991391, 771411.6543514762, 413, 9, 1568, 150558.91354772617, 598807.7361573568]

TABLE2.xlsx
Shape: (71, 2)

Row 1:
['Descriptive 2022 (MN Phas

In [21]:
# ============================================
# IDENTIFY MUNICIPALITY COLUMNS IN TABLE1
# ============================================

table1 = pd.read_excel(
    TABLES_DIR / "TABLE1.xlsx",
    header=None
)

geography_row = table1.iloc[8]
value_row = table1.iloc[10]

print("TABLE1 shape:", table1.shape)

print("\nFirst 30 geography entries:")
for i, value in enumerate(geography_row.iloc[:30]):
    print(i, "->", value)

print("\nFirst 30 corresponding values:")
for i, value in enumerate(value_row.iloc[:30]):
    print(i, "->", value)

TABLE1 shape: (16, 514)

First 30 geography entries:
0 -> Province, district and municipality
1 -> KwaZulu-Natal
2 -> nan
3 -> nan
4 -> nan
5 -> nan
6 -> nan
7 -> nan
8 -> nan
9 -> nan
10 -> Ugu
11 -> nan
12 -> nan
13 -> nan
14 -> nan
15 -> nan
16 -> nan
17 -> nan
18 -> nan
19 -> Umdoni Local Municipality
20 -> nan
21 -> nan
22 -> nan
23 -> nan
24 -> nan
25 -> nan
26 -> nan
27 -> nan
28 -> Umzumbe Local Municipality
29 -> nan

First 30 corresponding values:
0 -> nan
1 -> 12279374.662785625
2 -> 7768
3 -> 93
4 -> 136671
5 -> 6060765.093845516
6 -> 5819613.969936846
7 -> 543527.5990030895
8 -> 5919216.636794859
9 -> 6504690.025991391
10 -> 771411.6543514762
11 -> 413
12 -> 9
13 -> 1568
14 -> 150558.91354772617
15 -> 598807.7361573568
16 -> 24035.00464634987
17 -> 366400.6989871926
18 -> 407000.95536423894
19 -> 156080.8802611416
20 -> 99
21 -> 1
22 -> 262
23 -> 39850.27142586097
24 -> 104607.2186177397
25 -> 11985.390217549902
26 -> 74391.52150339381
27 -> 82051.35875776583
28 -> 138859.

In [22]:
# ============================================
# INSPECT TABLE1 COLUMN HIERARCHY
# ============================================

table1 = pd.read_excel(
    TABLES_DIR / "TABLE1.xlsx",
    header=None
)

for col in range(0, 80):

    values = []

    for row in [8, 9, 10]:
        value = table1.iloc[row, col]

        if pd.notna(value):
            values.append(str(value))

    if values:
        print(f"Column {col}:")
        for value in values:
            print("   ", value)

Column 0:
    Province, district and municipality
    Questionnaire type then Geography type then Sex
Column 1:
    KwaZulu-Natal
    Household
    12279374.662785625
Column 2:
    Homeless
    7768
Column 3:
    Transient
    93
Column 4:
    Institution
    136671
Column 5:
    Urban area
    6060765.093845516
Column 6:
    Tribal or Traditional area
    5819613.969936846
Column 7:
    Farm area
    543527.5990030895
Column 8:
    Male
    5919216.636794859
Column 9:
    Female
    6504690.025991391
Column 10:
    Ugu
    Household
    771411.6543514762
Column 11:
    Homeless
    413
Column 12:
    Transient
    9
Column 13:
    Institution
    1568
Column 14:
    Urban area
    150558.91354772617
Column 15:
    Tribal or Traditional area
    598807.7361573568
Column 16:
    Farm area
    24035.00464634987
Column 17:
    Male
    366400.6989871926
Column 18:
    Female
    407000.95536423894
Column 19:
    Umdoni Local Municipality
    Household
    156080.8802611416
Column 20:
    

In [23]:
# ============================================
# EXTRACT GEOGRAPHIC HIERARCHY FROM TABLE1
# ============================================

table1 = pd.read_excel(
    TABLES_DIR / "TABLE1.xlsx",
    header=None
)

geography_row = table1.iloc[8]

# Print every non-empty geography heading
print("Geographic headings found in TABLE1:\n")

for col, value in geography_row.items():

    if pd.notna(value):
        print(f"Column {col}: {value}")

Geographic headings found in TABLE1:

Column 0: Province, district and municipality
Column 1: KwaZulu-Natal
Column 10: Ugu
Column 19: Umdoni Local Municipality
Column 28: Umzumbe Local Municipality
Column 37: UMuziwabantu Local Municipality
Column 46: Ray Nkonyeni Local Municipality
Column 55: Umgungundlovu
Column 64: uMshwathi Local Municipality
Column 73: uMngeni Local Municipality
Column 82: Mpofana Local Municipality
Column 91: Impendle Local Municipality
Column 100: The Msunduzi Local Municipality
Column 109: Mkhambathini Local Municipality
Column 118: Richmond Local Municipality
Column 127: Uthukela
Column 136: Okhahlamba Local Municipality
Column 145: Inkosi Langalibalele Local Municipality
Column 154: Alfred Duma Local Municipality
Column 163: Umzinyathi
Column 172: Endumeni Local Municipality
Column 181: Nqutu Local Municipality
Column 190: Msinga Local Municipality
Column 199: Umvoti Local Municipality
Column 208: Amajuba
Column 217: Newcastle Local Municipality
Column 226: E

In [24]:
# ============================================
# CHECK TABLE2 TO TABLE5 STRUCTURE
# ============================================

for table_number in range(2, 6):

    file = TABLES_DIR / f"TABLE{table_number}.xlsx"

    df = pd.read_excel(
        file,
        header=None
    )

    print(f"\n{'=' * 60}")
    print(f"TABLE{table_number}")
    print(f"{'=' * 60}")

    print("Shape:", df.shape)

    # Show non-empty cells from the first 10 rows
    for row in range(min(10, len(df))):

        values = df.iloc[row].dropna().tolist()

        if values:
            print(f"\nRow {row}:")
            print(values[:12])


TABLE2
Shape: (71, 2)

Row 1:
['Descriptive 2022 (MN Phase 2)']

Row 2:
['Province, district and municipality']

Row 3:
['Counting: Population']

Row 5:
['Filters:']

Row 6:
['Default Summation', 'Population']

Row 8:
['Province, district and municipality']

Row 9:
['KwaZulu-Natal', 12423906.662785623]

TABLE3
Shape: (174, 2)

Row 1:
['Descriptive 2022 (MN Phase 2)']

Row 2:
['Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken']

Row 3:
['Counting: Population']

Row 5:
['Filters:']

Row 6:
['Default Summation', 'Population']

Row 8:
['Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken']

Row 9:
['KwaZulu-Natal', 12423906.662782008]

TABLE4
Shape: (15, 98)

Row 1:
['Descriptive 2022 (MN Phase 2)']

Row 2:
['Province, district and municipality then Attendance at an ECD institution then Attendance \xa0and educational institution then High

In [25]:
# ============================================
# VIEW TABLE2 COMPLETELY
# ============================================

table2 = pd.read_excel(
    TABLES_DIR / "TABLE2.xlsx",
    header=None
)

print("TABLE2 shape:", table2.shape)

display(table2)

TABLE2 shape: (71, 2)


,0,1
0,NaN,NaN
1,Descriptive 2022 (MN Phase 2),NaN
2,"Province, district and municipality",NaN
3,Counting: Population,NaN
4,NaN,NaN
...,...,...
66,NaN,NaN
67,NaN,NaN
68,NaN,NaN
69,NaN,NaN


In [26]:
# ============================================
# INSPECT TABLE2 EXCEL STRUCTURE
# ============================================

from openpyxl import load_workbook

table2_file = TABLES_DIR / "TABLE2.xlsx"

workbook = load_workbook(
    table2_file,
    data_only=True
)

sheet = workbook.active

print("Sheet:", sheet.title)
print("Rows:", sheet.max_row)
print("Columns:", sheet.max_column)

print("\nMerged cell ranges:")
for merged_range in sheet.merged_cells.ranges:
    print("-", merged_range)

Sheet: Data Sheet 0
Rows: 71
Columns: 2

Merged cell ranges:


C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


In [27]:
# ============================================
# SHOW ALL NON-EMPTY CELLS IN TABLE2
# ============================================

for row in range(1, sheet.max_row + 1):

    values = []

    for col in range(1, sheet.max_column + 1):
        value = sheet.cell(row=row, column=col).value

        if value is not None:
            values.append(f"Column {col}: {value}")

    if values:
        print(f"Row {row}:")
        for value in values:
            print("   ", value)

Row 2:
    Column 1: Descriptive 2022 (MN Phase 2)
Row 3:
    Column 1: Province, district and municipality
Row 4:
    Column 1: Counting: Population
Row 6:
    Column 1: Filters:
Row 7:
    Column 1: Default Summation
    Column 2: Population
Row 9:
    Column 1: Province, district and municipality
Row 10:
    Column 1: KwaZulu-Natal
    Column 2: 12423906.662785623
Row 11:
    Column 1: Ugu
    Column 2: 773401.6543514762
Row 12:
    Column 1: Umdoni Local Municipality
    Column 2: 156442.8802611416
Row 13:
    Column 1: Umzumbe Local Municipality
    Column 2: 139044.8059699663
Row 14:
    Column 1: UMuziwabantu Local Municipality
    Column 2: 115779.52551139316
Row 15:
    Column 1: Ray Nkonyeni Local Municipality
    Column 2: 362134.44260897517
Row 16:
    Column 1: Umgungundlovu
    Column 2: 1235715.497128913
Row 17:
    Column 1: uMshwathi Local Municipality
    Column 2: 118477.83272986031
Row 18:
    Column 1: uMngeni Local Municipality
    Column 2: 105068.59563358407
Row

In [28]:
# ============================================
# INSPECT TABLE3
# ============================================

table3 = pd.read_excel(
    TABLES_DIR / "TABLE3.xlsx",
    header=None
)

print("TABLE3 shape:", table3.shape)

for row in range(len(table3)):

    values = table3.iloc[row].dropna().tolist()

    if values:
        print(f"\nRow {row}:")
        print(values[:15])

TABLE3 shape: (174, 2)

Row 1:
['Descriptive 2022 (MN Phase 2)']

Row 2:
['Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken']

Row 3:
['Counting: Population']

Row 5:
['Filters:']

Row 6:
['Default Summation', 'Population']

Row 8:
['Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken']

Row 9:
['KwaZulu-Natal', 12423906.662782008]

Row 10:
['Male', 30078756.68715917]

Row 11:
['Female', 31948745.932222202]

Row 12:
['0', 1250353.0477566998]

Row 13:
['1', 1168240.7098342404]

Row 14:
['2', 1188298.519725562]

Row 15:
['3', 1130328.801133854]

Row 16:
['4', 1096293.544273597]

Row 17:
['5', 954477.8158382007]

Row 18:
['6', 1011729.9181077147]

Row 19:
['7', 1048430.4954019606]

Row 20:
['8', 1038061.8678377874]

Row 21:
['9', 1056403.960507052]

Row 22:
['10', 1055068.457665082]

Row 23:
['11', 1094814.8671509093]

Row 24:
['12', 1076

In [29]:
# ============================================
# INSPECT TABLE4 AND TABLE5
# ============================================

for table_number in [4, 5]:

    file = TABLES_DIR / f"TABLE{table_number}.xlsx"

    df = pd.read_excel(
        file,
        header=None
    )

    print(f"\n{'=' * 60}")
    print(f"TABLE{table_number}")
    print(f"{'=' * 60}")

    print("Shape:", df.shape)

    for row in range(min(12, len(df))):

        values = df.iloc[row].dropna().tolist()

        if values:
            print(f"\nRow {row}:")
            print(values[:15])


TABLE4
Shape: (15, 98)

Row 1:
['Descriptive 2022 (MN Phase 2)']

Row 2:
['Province, district and municipality then Attendance at an ECD institution then Attendance \xa0and educational institution then Highest level of education']

Row 3:
['Counting: Population']

Row 5:
['Filters:']

Row 6:
['Default Summation', 'Population']

Row 8:
['Province, district and municipality then Attendance at an ECD institution then Attendance \xa0and educational institution then Highest level of education', 'Umdoni Local Municipality', 'Umzumbe Local Municipality', 'UMuziwabantu Local Municipality', 'Ray Nkonyeni Local Municipality', 'Okhahlamba Local Municipality', 'Inkosi Langalibalele Local Municipality', 'Alfred Duma Local Municipality', 'Endumeni Local Municipality', 'Nqutu Local Municipality', 'Msinga Local Municipality', 'Umvoti Local Municipality', 'Zululand', 'eDumbe Local Municipality', 'UPhongolo Local Municipality']

Row 9:
[156442.8802611416, 139044.8059699663, 115779.52551139316, 362134.4

In [30]:
# ============================================
# INSPECT TABLE4 STRUCTURE
# ============================================

table4_file = TABLES_DIR / "TABLE4.xlsx"

table4 = pd.read_excel(
    table4_file,
    header=None
)

print("TABLE4 shape:", table4.shape)

# Show every non-empty value in the first 10 rows,
# together with its column number.
for row in range(10):

    print(f"\n--- ROW {row} ---")

    for col in range(table4.shape[1]):

        value = table4.iloc[row, col]

        if pd.notna(value):
            print(f"Column {col}: {value}")

TABLE4 shape: (15, 98)

--- ROW 0 ---

--- ROW 1 ---
Column 0: Descriptive 2022 (MN Phase 2)

--- ROW 2 ---
Column 0: Province, district and municipality then Attendance at an ECD institution then Attendance  and educational institution then Highest level of education

--- ROW 3 ---
Column 0: Counting: Population

--- ROW 4 ---

--- ROW 5 ---
Column 0: Filters:

--- ROW 6 ---
Column 0: Default Summation
Column 1: Population

--- ROW 7 ---

--- ROW 8 ---
Column 0: Province, district and municipality then Attendance at an ECD institution then Attendance  and educational institution then Highest level of education
Column 1: Umdoni Local Municipality
Column 2: Umzumbe Local Municipality
Column 3: UMuziwabantu Local Municipality
Column 4: Ray Nkonyeni Local Municipality
Column 5: Okhahlamba Local Municipality
Column 6: Inkosi Langalibalele Local Municipality
Column 7: Alfred Duma Local Municipality
Column 8: Endumeni Local Municipality
Column 9: Nqutu Local Municipality
Column 10: Msinga Lo

In [31]:
# ============================================
# TABLE4 — SHOW MUNICIPALITY/CATEGORY HEADERS
# ============================================

table4 = pd.read_excel(
    TABLES_DIR / "TABLE4.xlsx",
    header=None
)

for row in [8, 9, 10]:

    print(f"\n{'=' * 60}")
    print(f"ROW {row}")
    print(f"{'=' * 60}")

    for col in range(table4.shape[1]):

        value = table4.iloc[row, col]

        if pd.notna(value):
            print(f"{col}: {value}")


ROW 8
0: Province, district and municipality then Attendance at an ECD institution then Attendance  and educational institution then Highest level of education
1: Umdoni Local Municipality
2: Umzumbe Local Municipality
3: UMuziwabantu Local Municipality
4: Ray Nkonyeni Local Municipality
5: Okhahlamba Local Municipality
6: Inkosi Langalibalele Local Municipality
7: Alfred Duma Local Municipality
8: Endumeni Local Municipality
9: Nqutu Local Municipality
10: Msinga Local Municipality
11: Umvoti Local Municipality
12: Zululand
13: eDumbe Local Municipality
14: UPhongolo Local Municipality
15: Abaqulusi Local Municipality
16: Nongoma Local Municipality
17: Ulundi Local Municipality
18: Umkhanyakude
19: Umhlabuyalingana Local Municipality
20: Jozini Local Municipality
21: Mtubatuba Local Municipality
22: Big Five Hlabisa Local Municipality
23: King Cetshwayo
24: Mfolozi Local Municipality
25: uMhlathuze Local Municipality
26: uMlalazi Local Municipality
27: Mthonjaneni Local Municipality


In [32]:
# ============================================
# TABLE4 — INSPECT CATEGORIES AND VALUES
# ============================================

print("ROW 9")
print("=" * 60)

for col in range(table4.shape[1]):
    value = table4.iloc[9, col]

    if pd.notna(value):
        print(f"{col}: {value}")


print("\nROW 10")
print("=" * 60)

for col in range(table4.shape[1]):
    value = table4.iloc[10, col]

    if pd.notna(value):
        print(f"{col}: {value}")

ROW 9
1: 156442.8802611416
2: 139044.8059699663
3: 115779.52551139316
4: 362134.44260897517
5: 143132.47712344155
6: 230923.81823407355
7: 415035.5225078197
8: 100085.33395678402
9: 201132.54922641235
10: 206001.27131180197
11: 142041.82553754273
12: 942794.2047341896
13: 96735.251963954
14: 151540.592491773
15: 247262.85741333273
16: 225278.4029577534
17: 221977.09990736123
18: 738436.551107445
19: 191659.71423636732
20: 199152.94824647388
21: 215869.21386156185
22: 131754.67476304286
23: 1021343.5473776794
24: 159667.95534802426
25: 412074.9678163402
26: 241416.08102747172
27: 99288.7499040245
28: 108895.79328179012
29: 782660.810158101
30: 180939.49087498474
31: 324911.653516167
32: 165826.30055840104
33: 110983.36520852026
34: 563893.0973213947
35: 81676.11889501508
36: 133031.88088912296
37: 220619.69942617483
38: 128565.39811102043
39: 4239900.595478572
40: 4239900.595478572
41: 12423906.662782008
42: 2047864.4943416105
43: 417598.1223519458
44: 570022.6916401176
45: 346721.62163

In [33]:
# ============================================
# TABLE4 — INSPECT ROW LABELS
# ============================================

print("TABLE4 ROW LABELS")
print("=" * 60)

for row in range(8, table4.shape[0]):

    value = table4.iloc[row, 0]

    if pd.notna(value):
        print(f"Row {row}: {value}")

TABLE4 ROW LABELS
Row 8: Province, district and municipality then Attendance at an ECD institution then Attendance  and educational institution then Highest level of education
Row 14: © Statistics South Africa


In [34]:
# ============================================
# TABLE4 — INSPECT FIRST 8 COLUMNS
# ============================================

print("TABLE4 FIRST 8 COLUMNS")
print("=" * 70)

for row in range(table4.shape[0]):

    values = []

    for col in range(min(8, table4.shape[1])):
        value = table4.iloc[row, col]

        if pd.notna(value):
            values.append(f"C{col}={value}")

    if values:
        print(f"Row {row}:")
        print(" | ".join(values))

TABLE4 FIRST 8 COLUMNS
Row 1:
C0=Descriptive 2022 (MN Phase 2)
Row 2:
C0=Province, district and municipality then Attendance at an ECD institution then Attendance  and educational institution then Highest level of education
Row 3:
C0=Counting: Population
Row 5:
C0=Filters:
Row 6:
C0=Default Summation | C1=Population
Row 8:
C0=Province, district and municipality then Attendance at an ECD institution then Attendance  and educational institution then Highest level of education | C1=Umdoni Local Municipality | C2=Umzumbe Local Municipality | C3=UMuziwabantu Local Municipality | C4=Ray Nkonyeni Local Municipality | C5=Okhahlamba Local Municipality | C6=Inkosi Langalibalele Local Municipality | C7=Alfred Duma Local Municipality
Row 9:
C1=156442.8802611416 | C2=139044.8059699663 | C3=115779.52551139316 | C4=362134.44260897517 | C5=143132.47712344155 | C6=230923.81823407355 | C7=415035.5225078197
Row 14:
C0=© Statistics South Africa


In [35]:
# ============================================
# INSPECT TABLE3 STRUCTURE
# ============================================

table3 = pd.read_excel(
    TABLES_DIR / "TABLE3.xlsx",
    header=None
)

print("TABLE3 shape:", table3.shape)

for row in range(min(15, table3.shape[0])):

    print(f"\nROW {row}")
    print("-" * 60)

    for col in range(min(10, table3.shape[1])):

        value = table3.iloc[row, col]

        if pd.notna(value):
            print(f"C{col}: {value}")

TABLE3 shape: (174, 2)

ROW 0
------------------------------------------------------------

ROW 1
------------------------------------------------------------
C0: Descriptive 2022 (MN Phase 2)

ROW 2
------------------------------------------------------------
C0: Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken

ROW 3
------------------------------------------------------------
C0: Counting: Population

ROW 4
------------------------------------------------------------

ROW 5
------------------------------------------------------------
C0: Filters:

ROW 6
------------------------------------------------------------
C0: Default Summation
C1: Population

ROW 7
------------------------------------------------------------

ROW 8
------------------------------------------------------------
C0: Province, district and municipality then Sex then Age in completed years then Population group then Language most often s

In [36]:
# ============================================
# TABLE3 — SHOW ALL ROWS
# ============================================

print("TABLE3")
print("=" * 80)

for row in range(table3.shape[0]):

    col0 = table3.iloc[row, 0]
    col1 = table3.iloc[row, 1]

    print(f"Row {row}: C0={col0} | C1={col1}")

TABLE3
Row 0: C0=nan | C1=nan
Row 1: C0=Descriptive 2022 (MN Phase 2) | C1=nan
Row 2: C0=Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken | C1=nan
Row 3: C0=Counting: Population | C1=nan
Row 4: C0=nan | C1=nan
Row 5: C0=Filters: | C1=nan
Row 6: C0=Default Summation | C1=Population
Row 7: C0=nan | C1=nan
Row 8: C0=Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken | C1=nan
Row 9: C0=KwaZulu-Natal | C1=12423906.662782008
Row 10: C0=Male | C1=30078756.68715917
Row 11: C0=Female | C1=31948745.932222202
Row 12: C0=0 | C1=1250353.0477566998
Row 13: C0=1 | C1=1168240.7098342404
Row 14: C0=2 | C1=1188298.519725562
Row 15: C0=3 | C1=1130328.801133854
Row 16: C0=4 | C1=1096293.544273597
Row 17: C0=5 | C1=954477.8158382007
Row 18: C0=6 | C1=1011729.9181077147
Row 19: C0=7 | C1=1048430.4954019606
Row 20: C0=8 | C1=1038061.8678377874
Row 21: C0=9 

In [37]:
# ============================================
# TABLE1 — INSPECT ROW STRUCTURE
# ============================================

table1 = pd.read_excel(
    TABLES_DIR / "TABLE1.xlsx",
    header=None
)

print("TABLE1 shape:", table1.shape)

print("\nROW LABELS / FIRST 5 COLUMNS")
print("=" * 80)

for row in range(min(16, table1.shape[0])):

    values = []

    for col in range(min(5, table1.shape[1])):

        value = table1.iloc[row, col]

        if pd.notna(value):
            values.append(f"C{col}={value}")

    if values:
        print(f"Row {row}: " + " | ".join(values))

TABLE1 shape: (16, 514)

ROW LABELS / FIRST 5 COLUMNS
Row 1: C0=Descriptive 2022 (MN Phase 2)
Row 2: C0=Province, district and municipality and Questionnaire type then Geography type then Sex
Row 3: C0=Counting: Population
Row 5: C0=Filters:
Row 6: C0=Default Summation | C1=Population
Row 8: C0=Province, district and municipality | C1=KwaZulu-Natal
Row 9: C0=Questionnaire type then Geography type then Sex | C1=Household | C2=Homeless | C3=Transient | C4=Institution
Row 10: C1=12279374.662785625 | C2=7768 | C3=93 | C4=136671
Row 15: C0=© Statistics South Africa


In [38]:
# ============================================
# TABLE1 — LIST ALL GEOGRAPHY HEADINGS
# ============================================

print("TABLE1 GEOGRAPHY HEADINGS")
print("=" * 80)

for col in range(table1.shape[1]):

    value = table1.iloc[8, col]

    if pd.notna(value):
        print(f"Column {col}: {value}")
        

TABLE1 GEOGRAPHY HEADINGS
Column 0: Province, district and municipality
Column 1: KwaZulu-Natal
Column 10: Ugu
Column 19: Umdoni Local Municipality
Column 28: Umzumbe Local Municipality
Column 37: UMuziwabantu Local Municipality
Column 46: Ray Nkonyeni Local Municipality
Column 55: Umgungundlovu
Column 64: uMshwathi Local Municipality
Column 73: uMngeni Local Municipality
Column 82: Mpofana Local Municipality
Column 91: Impendle Local Municipality
Column 100: The Msunduzi Local Municipality
Column 109: Mkhambathini Local Municipality
Column 118: Richmond Local Municipality
Column 127: Uthukela
Column 136: Okhahlamba Local Municipality
Column 145: Inkosi Langalibalele Local Municipality
Column 154: Alfred Duma Local Municipality
Column 163: Umzinyathi
Column 172: Endumeni Local Municipality
Column 181: Nqutu Local Municipality
Column 190: Msinga Local Municipality
Column 199: Umvoti Local Municipality
Column 208: Amajuba
Column 217: Newcastle Local Municipality
Column 226: Emadlangeni L

In [39]:
# ============================================
# TABLE1 — COMPLETE GEOGRAPHY LIST
# ============================================

geography_headings = []

for col in range(table1.shape[1]):
    
    value = table1.iloc[8, col]
    
    if pd.notna(value):
        geography_headings.append((col, str(value)))

print("Total geography headings:", len(geography_headings))
print("=" * 80)

for number, (col, name) in enumerate(geography_headings, start=1):
    print(f"{number:02d}. Column {col}: {name}")

Total geography headings: 58
01. Column 0: Province, district and municipality
02. Column 1: KwaZulu-Natal
03. Column 10: Ugu
04. Column 19: Umdoni Local Municipality
05. Column 28: Umzumbe Local Municipality
06. Column 37: UMuziwabantu Local Municipality
07. Column 46: Ray Nkonyeni Local Municipality
08. Column 55: Umgungundlovu
09. Column 64: uMshwathi Local Municipality
10. Column 73: uMngeni Local Municipality
11. Column 82: Mpofana Local Municipality
12. Column 91: Impendle Local Municipality
13. Column 100: The Msunduzi Local Municipality
14. Column 109: Mkhambathini Local Municipality
15. Column 118: Richmond Local Municipality
16. Column 127: Uthukela
17. Column 136: Okhahlamba Local Municipality
18. Column 145: Inkosi Langalibalele Local Municipality
19. Column 154: Alfred Duma Local Municipality
20. Column 163: Umzinyathi
21. Column 172: Endumeni Local Municipality
22. Column 181: Nqutu Local Municipality
23. Column 190: Msinga Local Municipality
24. Column 199: Umvoti Local 

In [40]:
# ============================================
# TABLE1 — IDENTIFY MUNICIPALITY BLOCKS
# ============================================

municipality_columns = []

for col in range(1, table1.shape[1], 9):

    geography = table1.iloc[8, col]

    if pd.notna(geography):
        geography = str(geography)

        # Municipality names in this export contain "Municipality"
        if "Municipality" in geography:
            municipality_columns.append((col, geography))

print("Municipalities found:", len(municipality_columns))
print("=" * 80)

for col, municipality in municipality_columns:
    print(f"Column {col}: {municipality}")

Municipalities found: 44
Column 19: Umdoni Local Municipality
Column 28: Umzumbe Local Municipality
Column 37: UMuziwabantu Local Municipality
Column 46: Ray Nkonyeni Local Municipality
Column 64: uMshwathi Local Municipality
Column 73: uMngeni Local Municipality
Column 82: Mpofana Local Municipality
Column 91: Impendle Local Municipality
Column 100: The Msunduzi Local Municipality
Column 109: Mkhambathini Local Municipality
Column 118: Richmond Local Municipality
Column 136: Okhahlamba Local Municipality
Column 145: Inkosi Langalibalele Local Municipality
Column 154: Alfred Duma Local Municipality
Column 172: Endumeni Local Municipality
Column 181: Nqutu Local Municipality
Column 190: Msinga Local Municipality
Column 199: Umvoti Local Municipality
Column 217: Newcastle Local Municipality
Column 226: Emadlangeni Local Municipality
Column 235: Dannhauser Local Municipality
Column 253: eDumbe Local Municipality
Column 262: UPhongolo Local Municipality
Column 271: Abaqulusi Local Municipa

In [41]:
# ============================================
# TABLE1 — EXTRACT MUNICIPALITY DATA
# ============================================

# The category names are stored in Row 9.
category_names = [
    str(table1.iloc[9, col])
    for col in range(1, 9)
]

print("Categories:")
for category in category_names:
    print("-", category)

# Store extracted municipality records
municipality_records = []

for start_col, municipality in municipality_columns:

    record = {
        "municipality": municipality
    }

    # Each municipality has 8 category columns
    for offset, category in enumerate(category_names, start=1):

        value = table1.iloc[10, start_col + offset]

        record[category] = value

    municipality_records.append(record)

table1_municipalities = pd.DataFrame(municipality_records)

print("\nExtracted shape:", table1_municipalities.shape)

print("\nFirst 5 municipalities:")
display(table1_municipalities.head())

Categories:
- Household
- Homeless
- Transient
- Institution
- Urban area
- Tribal or Traditional area
- Farm area
- Male

Extracted shape: (44, 9)

First 5 municipalities:


,municipality,Household,Homeless,Transient,Institution,Urban area,Tribal or Traditional area,Farm area,Male
0,Umdoni Local Municipality,99,1,262,39850.271426,104607.218618,11985.390218,74391.521503,82051.358758
1,Umzumbe Local Municipality,56,4,125,0.000000,139029.363876,15.442094,65512.792209,73532.013761
2,UMuziwabantu Local Municipality,54,3,183,12871.211719,100534.222489,2374.091303,54516.210659,61263.314852
3,Ray Nkonyeni Local Municipality,204,1,998,97837.430403,254636.931174,9660.081033,171980.174616,190154.267993
4,uMshwathi Local Municipality,10,1,605,15332.360949,72093.563278,31051.908503,55836.571936,62641.260794


In [42]:
# ============================================
# TABLE1 — VERIFY EXTRACTED COLUMNS
# ============================================

print("Extracted columns:")
print("=" * 80)

for number, column in enumerate(table1_municipalities.columns, start=1):
    print(f"{number}. {column}")

print("\nShape:", table1_municipalities.shape)

Extracted columns:
1. municipality
2. Household
3. Homeless
4. Transient
5. Institution
6. Urban area
7. Tribal or Traditional area
8. Farm area
9. Male

Shape: (44, 9)


In [43]:
# ============================================
# TABLE2 — EXTRACT POPULATION BY GEOGRAPHY
# ============================================

table2 = pd.read_excel(
    TABLES_DIR / "TABLE2.xlsx",
    header=None
)

# Data starts at row 10.
# Row 10 onward contains geography names and population values.
table2_population = table2.iloc[10:, [0, 1]].copy()

# Give the columns clear names
table2_population.columns = [
    "geography",
    "population"
]

# Remove empty rows
table2_population = table2_population.dropna(
    subset=["geography"]
)

# Remove the Stats SA copyright/footer
table2_population = table2_population[
    table2_population["geography"] != "© Statistics South Africa"
]

print("TABLE2 extracted shape:", table2_population.shape)

display(table2_population.head(15))

TABLE2 extracted shape: (56, 2)


,geography,population
10,Ugu,773401.654351
11,Umdoni Local Municipality,156442.880261
12,Umzumbe Local Municipality,139044.80597
13,UMuziwabantu Local Municipality,115779.525511
14,Ray Nkonyeni Local Municipality,362134.442609
15,Umgungundlovu,1235715.497129
16,uMshwathi Local Municipality,118477.83273
17,uMngeni Local Municipality,105068.595634
18,Mpofana Local Municipality,33382.364261
19,Impendle Local Municipality,36647.723855


In [44]:
# ============================================
# MERGE TABLE1 + TABLE2
# ============================================

# Keep only TABLE2 rows that match municipalities
# already identified in TABLE1.
table2_municipalities = table2_population[
    table2_population["geography"].isin(
        table1_municipalities["municipality"]
    )
].copy()

# Rename geography so both tables use the same key
table2_municipalities = table2_municipalities.rename(
    columns={"geography": "municipality"}
)

# Merge the two municipality-level tables
statssa_merged = table1_municipalities.merge(
    table2_municipalities,
    on="municipality",
    how="left",
    validate="one_to_one"
)

print("Merged shape:", statssa_merged.shape)

print("\nMerged columns:")
print(list(statssa_merged.columns))

print("\nFirst 5 rows:")
display(statssa_merged.head())

Merged shape: (44, 10)

Merged columns:
['municipality', 'Household', 'Homeless', 'Transient', 'Institution', 'Urban area', 'Tribal or Traditional area', 'Farm area', 'Male', 'population']

First 5 rows:


,municipality,Household,Homeless,Transient,Institution,Urban area,Tribal or Traditional area,Farm area,Male,population
0,Umdoni Local Municipality,99,1,262,39850.271426,104607.218618,11985.390218,74391.521503,82051.358758,156442.880261
1,Umzumbe Local Municipality,56,4,125,0.000000,139029.363876,15.442094,65512.792209,73532.013761,139044.80597
2,UMuziwabantu Local Municipality,54,3,183,12871.211719,100534.222489,2374.091303,54516.210659,61263.314852,115779.525511
3,Ray Nkonyeni Local Municipality,204,1,998,97837.430403,254636.931174,9660.081033,171980.174616,190154.267993,362134.442609
4,uMshwathi Local Municipality,10,1,605,15332.360949,72093.563278,31051.908503,55836.571936,62641.260794,118477.83273


In [45]:
# ============================================
# VERIFY THE MERGED DATASET
# ============================================

print("Number of municipalities:", len(statssa_merged))

print("\nMissing population values:")
print(statssa_merged["population"].isna().sum())

print("\nDuplicate municipalities:")
print(statssa_merged["municipality"].duplicated().sum())

print("\nMerged dataset shape:")
print(statssa_merged.shape)

print("\nMissing values by column:")
print(statssa_merged.isna().sum())

Number of municipalities: 44

Missing population values:
0

Duplicate municipalities:
0

Merged dataset shape:
(44, 10)

Missing values by column:
municipality                  0
Household                     0
Homeless                      0
Transient                     0
Institution                   0
Urban area                    0
Tribal or Traditional area    0
Farm area                     0
Male                          0
population                    0
dtype: int64


In [47]:
# ============================================
# SAVE VERIFIED STATSSA MERGED DATASET
# ============================================

from pathlib import Path

# Project folders
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Stats SA processed-data folder
STATSSA_PROCESSED_DIR = PROCESSED_DIR / "statssa"
STATSSA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Output file
STATSSA_MERGED_FILE = (
    STATSSA_PROCESSED_DIR
    / "kzn_municipality_population_demographics.csv"
)

# Save verified merged dataset
statssa_merged.to_csv(
    STATSSA_MERGED_FILE,
    index=False
)

print("Saved successfully:")
print(STATSSA_MERGED_FILE)

Saved successfully:
c:\Users\Admin\Desktop\Dirisa_1\SPU-TEAM-DIRISA\data\processed\statssa\kzn_municipality_population_demographics.csv


In [48]:
# ============================================
# TABLE3 — INSPECT ALL ROWS
# ============================================

print("TABLE3 COMPLETE STRUCTURE")
print("=" * 80)

for row in range(table3.shape[0]):

    geography_or_category = table3.iloc[row, 0]
    value = table3.iloc[row, 1]

    if pd.notna(geography_or_category) or pd.notna(value):
        print(
            f"Row {row:03d} | "
            f"Category: {geography_or_category} | "
            f"Value: {value}"
        )

TABLE3 COMPLETE STRUCTURE
Row 001 | Category: Descriptive 2022 (MN Phase 2) | Value: nan
Row 002 | Category: Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken | Value: nan
Row 003 | Category: Counting: Population | Value: nan
Row 005 | Category: Filters: | Value: nan
Row 006 | Category: Default Summation | Value: Population
Row 008 | Category: Province, district and municipality then Sex then Age in completed years then Population group then Language most often spoken | Value: nan
Row 009 | Category: KwaZulu-Natal | Value: 12423906.662782008
Row 010 | Category: Male | Value: 30078756.68715917
Row 011 | Category: Female | Value: 31948745.932222202
Row 012 | Category: 0 | Value: 1250353.0477566998
Row 013 | Category: 1 | Value: 1168240.7098342404
Row 014 | Category: 2 | Value: 1188298.519725562
Row 015 | Category: 3 | Value: 1130328.801133854
Row 016 | Category: 4 | Value: 1096293.544273597
Row 017 | Category: 5